# download_checkpoint

Paste a Hugging Face link, run every cell, and the file lands in this repo's `checkpoints/` folder, at the place the
pipeline looks for it. Use it to pull one checkpoint for testing (a resume, a branch, `src.rerun`, an ask-only probe)
without downloading a whole run.

Any of these links works, copied from the browser:

| link | what is downloaded |
|---|---|
| `https://huggingface.co/<user>/<repo>/blob/main/checkpoints/<run>/decision/step-1500/adapter_model.safetensors` | that file, and with `WHOLE_CHECKPOINT = True` the rest of its step folder (an adapter needs `adapter_config.json` beside the weights) |
| `.../resolve/main/...` (a download link, `?download=true` or not) | the same |
| `https://huggingface.co/<user>/<repo>/tree/main/checkpoints/<run>/decision/step-1500` (a folder page) | every file in that folder |
| `.../tree/main/checkpoints/<run>` | every checkpoint of the run (can be large: 12 steps of an 8B adapter is about 1 GB) |

The Hub repo mirrors this tree, so `checkpoints/<run>/decision/step-1500/...` on the Hub is written to
`checkpoints/<run>/decision/step-1500/...` here. The run's `results/<run>/settings.json` comes too (`ALSO_SETTINGS`), as
`src/persist.py pull_branch_source` does, because a resume or a branch reads it. Files already here are kept unless
`OVERWRITE = True`. Every file is checked against the Hub's SHA-256 at the end.

A public repo needs no token. For a private one, set `HF_TOKEN` in the environment (or Colab's secrets) first.

Edit the first code cell and nothing else.

In [ ]:
# One link per line. Copy them from the Hugging Face page (a file's page, its download link, or a folder page).
# A run's .../decision folder is every checkpoint of the run (E-negative's: 12 steps, 38 files, 1.2 GB);
# a .../decision/step-1500 folder is one checkpoint (about 90 MB for an 8B adapter).
LINKS = [
    "https://huggingface.co/adamchanna/bdp-plunkett-qwen3/tree/main/checkpoints/20261116-221120_paper_minimum_threshold_Qwen3-8B_e-negative/decision",
]

WHOLE_CHECKPOINT = True   # a link to one file inside a step-<N> folder downloads the whole step folder
ALSO_SETTINGS = True      # also fetch results/<run>/settings.json, which a resume or a branch reads
OVERWRITE = False         # True: replace files that are already here
DRY_RUN = False           # True: list what each link would download, and its size, without downloading
REPO_ROOT = None          # None: this repo (found from where the notebook runs); or a path

## Set up

In [ ]:
# Find the repo and make sure huggingface_hub is installed.
import os, subprocess, sys
from pathlib import Path


def find_repo_root(given=None):
    """The folder holding src/ and configs/: the given one, the current folder or a parent, or Colab's clone."""
    if given:
        return Path(given).resolve()
    here = Path.cwd().resolve()
    for p in [here, *here.parents, Path("/content/Decision_Task_Database_Experiments")]:
        if (p / "src").is_dir() and (p / "configs").is_dir():
            return p
    raise SystemExit("could not find the repo (a folder with src/ and configs/): set REPO_ROOT")


try:
    import huggingface_hub  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)

ROOT = find_repo_root(REPO_ROOT)
TOKEN = os.environ.get("HF_TOKEN") or None
if TOKEN is None:
    try:                                     # Colab's secrets panel, when the notebook runs there
        from google.colab import userdata
        TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
print("repo:", ROOT)
print("token:", "set" if TOKEN else "none (fine for a public repo)")

## Download

In [ ]:
# Read each link, list what it names on the Hub, and copy those files into place.
import hashlib
import re
import shutil
import tempfile
from urllib.parse import unquote, urlparse

from huggingface_hub import HfApi, hf_hub_download

LINK = re.compile(r"^/(?:(?P<type>datasets|spaces)/)?(?P<repo>[^/]+/[^/]+)(?:/(?P<kind>blob|resolve|tree)/(?P<rev>[^/]+)(?:/(?P<path>.*))?)?/?$")
STEP = re.compile(r"^(?P<folder>.*/step-\d+)/[^/]+$")


def parse_link(link):
    """(repo_id, repo_type, revision, path, is_folder) for a huggingface.co link."""
    u = urlparse(link.strip())
    if u.netloc not in ("huggingface.co", "www.huggingface.co"):
        raise ValueError(f"not a huggingface.co link: {link}")
    m = LINK.match(unquote(u.path))
    if not m:
        raise ValueError(f"cannot read this link: {link}")
    repo_type = {"datasets": "dataset", "spaces": "space"}.get(m["type"], "model")
    path = (m["path"] or "").strip("/")
    return m["repo"], repo_type, m["rev"] or "main", path, m["kind"] in (None, "tree")


def files_for(every, repo_id, rev, path, is_folder, whole_checkpoint):
    """Every Hub file the link names (from `every`, the repo's file list): the file, or everything under the folder."""
    if not is_folder:
        if path not in every:
            raise SystemExit(f"{path} is not in {repo_id} at {rev}")
        step = STEP.match(path)
        if whole_checkpoint and step:
            path, is_folder = step["folder"], True
        else:
            return [path]
    prefix = f"{path}/" if path else ""
    got = sorted(f for f in every if f.startswith(prefix))
    if not got:
        raise SystemExit(f"nothing under {path or 'the repo root'} in {repo_id} at {rev}")
    return got


def run_of(path):
    """The run id of a checkpoints/<run>/... or results/<run>/... path, else None."""
    parts = path.split("/")
    return parts[1] if len(parts) > 2 and parts[0] in ("checkpoints", "results") else None


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def fetch(link):
    api = HfApi()
    repo_id, repo_type, rev, path, is_folder = parse_link(link)
    every = api.list_repo_files(repo_id, repo_type=repo_type, revision=rev, token=TOKEN)   # listed once per link
    wanted = files_for(every, repo_id, rev, path, is_folder, WHOLE_CHECKPOINT)
    if not all(f.startswith("checkpoints/") for f in wanted):
        print(f"  note: not every file is under checkpoints/; each goes to the same path here as on the Hub")
    runs = {run_of(f) for f in wanted} - {None}
    if ALSO_SETTINGS:
        listed = set(every)
        wanted += sorted(s for s in (f"results/{r}/settings.json" for r in runs) if s in listed and s not in wanted)
    sizes = {i.path: i.size for i in api.get_paths_info(repo_id, wanted, repo_type=repo_type, revision=rev, token=TOKEN)}
    new = [f for f in wanted if OVERWRITE or not (ROOT / f).exists()]
    print(f"{link}\n  {repo_id} ({repo_type}) at {rev}: {len(wanted)} file(s), {sum(sizes.values()) / 1e9:.2f} GB; "
          f"{len(new)} to download ({sum(sizes[f] for f in new) / 1e9:.2f} GB), {len(wanted) - len(new)} already here")
    if DRY_RUN:
        for f in wanted:
            print(f"  {'would fetch' if f in new else 'have      '}  {f} ({sizes[f] / 1e6:.1f} MB)")
        return []
    done = []
    # Downloaded to a temporary folder and copied, so no .cache/huggingface folder is left inside checkpoints/.
    with tempfile.TemporaryDirectory() as tmp:
        for f in wanted:
            dest = ROOT / f
            if dest.exists() and not OVERWRITE:
                print(f"  kept     {f} (already here; OVERWRITE = True replaces it)")
                done.append((repo_id, repo_type, rev, f))
                continue
            local = hf_hub_download(repo_id, f, repo_type=repo_type, revision=rev, local_dir=tmp, token=TOKEN)
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(local, dest)
            print(f"  fetched  {f} ({dest.stat().st_size / 1e6:.1f} MB)")
            done.append((repo_id, repo_type, rev, f))
    return done


DOWNLOADED = []
for link in LINKS:
    DOWNLOADED += fetch(link)

## Check

In [ ]:
# Check every file against the Hub: size, and SHA-256 for the large (LFS) ones such as adapter_model.safetensors.
from collections import defaultdict

api = HfApi()
by_repo = defaultdict(list)
for repo_id, repo_type, rev, f in DOWNLOADED:
    by_repo[(repo_id, repo_type, rev)].append(f)
bad = 0
for (repo_id, repo_type, rev), files in by_repo.items():
    for info in api.get_paths_info(repo_id, files, repo_type=repo_type, revision=rev, token=TOKEN):
        here = ROOT / info.path
        ok = here.stat().st_size == info.size
        how = "size"
        if ok and getattr(info, "lfs", None):
            ok, how = sha256(here) == info.lfs.sha256, "sha256"
        bad += not ok
        print(f"  {'ok ' if ok else 'BAD'} ({how:6s}) {info.path}")
steps = sorted({str((ROOT / f).parent.relative_to(ROOT)) for *_, f in DOWNLOADED if "/step-" in f})
for s in steps:
    names = {p.name for p in (ROOT / s).iterdir()}
    usable = "adapter_config.json" in names and any(n.endswith(".safetensors") for n in names)
    print(f"  {s}: {'adapter_config.json and weights present' if usable else 'NOT a complete adapter: ' + ', '.join(sorted(names))}")
if DRY_RUN:
    print("dry run: nothing downloaded (set DRY_RUN = False to download)")
elif not DOWNLOADED:
    print("nothing downloaded")
else:
    print("all files match the Hub" if not bad else f"{bad} file(s) do not match the Hub: delete them and run again with OVERWRITE = True")

## Using what you downloaded

- **Branch from it:** set `model_hyperparameters.start_from` to `{"run_id": "<run>", "step": <N>}` in a settings file,
  or `python scripts/run_one_persona.py --settings <file> --start-from <run> <N>`.
- **Resume the run:** `RESUME_RUN_ID` and `RESUME_STEP` in `run_experiment.ipynb` (a resume also needs the run's
  results folder: download `.../tree/main/results/<run>` the same way).
- **Ask it questions without training:** `ask_only.ipynb`.

`checkpoints/` is not in git, so nothing here is committed.